# Czech Day-Ahead Electricity Market & Weather Dynamics
### Exploratory Data Analysis & Feature Justification

This notebook analyzes hourly spot prices from **OTE-CR** alongside regional weather patterns from **Open-Meteo** and renewable generation metrics.

**Core Questions Explored:**
1. What does the intraday price profile ("Duck Curve") look like across seasons?
2. How strong is the solar cannibalization effect during summer midday peaks?
3. Autocorrelation structure: why are $t-24$ and $t-168$ lags the dominant predictive signals?

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 10

## 1. Load Market & Feature Data from SQLite

In [ ]:
conn = sqlite3.connect('../data/energy.db')
df = pd.read_sql("""
    SELECT 
        timestamp_utc, 
        price_eur_mwh, 
        avg_temperature_c,
        avg_cloud_cover_pct,
        avg_wind_speed,
        avg_irradiance,
        hour_of_day,
        day_of_week,
        month,
        is_holiday,
        solar_generation_mw,
        wind_generation_mw
    FROM fact_hourly_market_features
    WHERE price_eur_mwh IS NOT NULL
    ORDER BY timestamp_utc
""", conn, parse_dates=['timestamp_utc'])

df.set_index('timestamp_utc', inplace=True)
print(f"Loaded {len(df):,} hourly market records from {df.index.min().date()} to {df.index.max().date()}")
df.head()

## 2. Intraday Price Distribution by Hour & Day of Week
Notice the classic two-peak pattern (morning industrial ramp at 07:00-09:00 UTC and evening domestic peak at 17:00-20:00 UTC).

In [ ]:
hourly_stats = df.groupby('hour_of_day')['price_eur_mwh'].agg(['median', lambda x: np.percentile(x, 25), lambda x: np.percentile(x, 75)])
hourly_stats.columns = ['Median', 'Q25', 'Q75']

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hourly_stats.index, hourly_stats['Median'], label='Median Price', color='#1f77b4', lw=2)
ax.fill_between(hourly_stats.index, hourly_stats['Q25'], hourly_stats['Q75'], color='#1f77b4', alpha=0.2, label='IQR (25-75%)')
ax.set_title('Hourly OTE Day-Ahead Spot Price Profile (EUR/MWh)', fontweight='bold')
ax.set_xlabel('Hour of Day (UTC)')
ax.set_ylabel('EUR / MWh')
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()

## 3. Solar Irradiance vs Midday Prices
Examining the merit-order effect: high solar irradiance during summer suppresses peak midday prices, occasionally causing negative prices.

In [ ]:
summer_midday = df[(df['month'].isin([5, 6, 7, 8])) & (df['hour_of_day'].isin([11, 12, 13]))]

fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(
    summer_midday['avg_irradiance'], 
    summer_midday['price_eur_mwh'], 
    c=summer_midday['avg_temperature_c'], 
    cmap='viridis', 
    alpha=0.4, 
    edgecolors='none'
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Mean Temperature (°C)')
ax.set_title('Summer Midday: Solar Irradiance vs Spot Price', fontweight='bold')
ax.set_xlabel('Average Irradiance (W/m²)')
ax.set_ylabel('Price (EUR/MWh)')
ax.axhline(0, color='red', linestyle='--', alpha=0.7, label='Zero price threshold')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Autocorrelation & Lag Structure
Price series in liberalized electricity markets show strong diurnal ($t-24$) and weekly ($t-168$) cyclical autocorrelation. This justifies our selection of autoregressive lag features.

In [ ]:
lags = [1, 2, 24, 48, 72, 168]
corrs = [df['price_eur_mwh'].corr(df['price_eur_mwh'].shift(l)) for l in lags]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([f't-{l}h' for l in lags], corrs, color='#2ca02c', alpha=0.8)
ax.set_title('Price Autocorrelation across Lag Horizons', fontweight='bold')
ax.set_ylabel('Pearson Correlation')
ax.set_ylim(0, 1.0)
for bar, corr in zip(bars, corrs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{corr:.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()